[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/06_unet_segmentation.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 06 — U-Net + ResNet-34: Segmentación Pixel-Level

A diferencia de los modelos anteriores (clasificación de parche), U-Net produce mapas de probabilidad 128×128 para detectar la localización exacta de cada deslizamiento.

---

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")


## 1. Arquitectura U-Net + ResNet-34

In [ ]:
from src.models import build_unet_resnet34, model_summary
import torch

model = build_unet_resnet34(n_channels=14, pretrained=True)
print(model_summary(model, n_channels=14))

# Test con entrada 128×128
dummy = torch.randn(2, 14, 128, 128)
out   = model(dummy)
print(f"\nForward pass: (2,14,128,128) → {out.shape}  ← (B,1,H,W) mapa pixel-level")
print("→ Cada píxel tiene una probabilidad de deslizamiento independiente.")

## 2. Dice Loss — importancia para objetos pequeños

In [ ]:
from src.train import DiceLoss, DiceBCELoss
import torch

dice_loss = DiceLoss()
bce_loss  = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([0.703]))
comb_loss = DiceBCELoss(dice_weight=0.5, pos_weight=0.703)

# Simular predicción sobre un parche pequeño (área ~2%)
H, W = 128, 128
true_mask = torch.zeros(1, 1, H, W)
true_mask[:, :, 60:64, 60:64] = 1.0   # 4×4 px = 0.098% del parche

# Predicción perfecta
logits_perfect = torch.full((1,1,H,W), -5.0)
logits_perfect[:,:,60:64,60:64] = 5.0

# Predicción errónea
logits_wrong = torch.full((1,1,H,W), -5.0)

print(f"Predicción perfecta:")
print(f"  Dice Loss: {dice_loss(logits_perfect, true_mask):.4f}")
print(f"  BCE Loss:  {bce_loss(logits_perfect, true_mask):.4f}")
print(f"  Comb Loss: {comb_loss(logits_perfect, true_mask):.4f}")
print(f"\nPredicción errónea (todo cero):")
print(f"  Dice Loss: {dice_loss(logits_wrong, true_mask):.4f}")
print(f"  BCE Loss:  {bce_loss(logits_wrong, true_mask):.4f}")
print(f"  Comb Loss: {comb_loss(logits_wrong, true_mask):.4f}")
print("\n→ El Dice Loss penaliza fuertemente la ausencia total de predicción.")

## 3. Entrenamiento K-Fold con segmentación

In [ ]:
# ⚠️ U-Net con segmentación es más lento (~14ms/img vs 8ms de ResNet-50)
summary = run_kfold(cfg)
print("\nEntrenamiento U-Net completado.")

## 4. Visualización de predicciones pixel-level

In [ ]:
import h5py
import matplotlib.pyplot as plt
from pathlib import Path
from src.utils import load_checkpoint, get_device, visualize_prediction
from src.models import build_unet_resnet34
from src.dataset import normalize_patch

device = get_device()
fold_idx = 0
ckpt_path = Path(f"../results/unet_resnet34/fold_{fold_idx}/best_model.pth")

if ckpt_path.exists():
    model = build_unet_resnet34(n_channels=14).to(device)
    load_checkpoint(model, str(ckpt_path), device=device)
    model.eval()
    
    # Cargar parche con deslizamiento
    img_dir  = Path("../data/TrainData/img")
    mask_dir = Path("../data/TrainData/mask")
    for f in sorted(img_dir.glob("*.h5")):
        mf = mask_dir / f.name
        with h5py.File(mf,"r") as hf: mask = hf[list(hf.keys())[0]][()]
        if mask.mean() > 0.01: break
    
    with h5py.File(f,"r") as hf: patch = hf[list(hf.keys())[0]][()].astype("float32")
    patch_n = normalize_patch(patch)
    tensor  = torch.from_numpy(patch_n.transpose(2,0,1)).unsqueeze(0).to(device)
    
    with torch.no_grad():
        pred_logit = model(tensor)
        pred_prob  = torch.sigmoid(pred_logit).squeeze().cpu().numpy()
    
    visualize_prediction(patch, pred_prob, mask, title=f"U-Net — {f.name}")
else:
    print("Checkpoint no encontrado. Ejecutar entrenamiento primero.")